# Project 2: Spark Class and Example
Jacob A. Fericy

This notebook completes creates a custom PySpark class called `SparkDataCheck` that wraps a Spark SQL DataFrame and adds methods for validation and summarization. The class then is designed to validation methods, modify the Spark DataFrame stored in the object and return the object itself, while summarization methods return regular pandas DataFrames.

After creating the class, I use both pandas-on-Spark and Spark SQL DataFrames to analyze NFL weekly quarterback data. The goal here is to produce the same basic season-level summaries using both APIs, compare the outputs, and note any differences in behavior.


In [ ]:
import pandas as pd

if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import importlib
import helpers_spark
importlib.reload(spark_data_check)

from helpers_spark import SparkDataCheck

spark = (
    SparkSession.builder
    .appName("Project2")
    .master("local[*]")
    .getOrCreate()
)

ps.set_option("compute.ops_on_diff_frames", True)


# Part I: Creating and testing the `SparkDataCheck` class

For the first part of the project, I use the class method that reads a CSV file into Spark and returns an instance of that class. With the air quality data being the same dataset used earlier in the course, it is a good test case because it contains both numeric and non-numeric fields and allows me to show the required range checks, level checks, missingness checks, and summary methods.

Because the air-quality CSV is hosted at a web URL, the updated `from_csv()` method first downloads the file locally when a URL is supplied and then uses `spark.read.load()` on the local CSV. This keeps the class aligned with the project requirement while avoiding Spark's `https` filesystem issue.

In [ ]:
air_path = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

air_obj = SparkDataCheck.from_csv(spark, air_path)
air_obj.df.show(5)

#print schema
air_obj.df.printSchema()

## Testing `check_numeric_range()`

This method should:
1. Work on numeric columns only
2. Allow a lower bound, an upper bound, or both
3. Return `NULL` when the original value is `NULL`
4. Print a message and leave the object unchanged if the column is not numeric
5. Print a message if neither bound is supplied

I provide five examples below to cover the normal cases and the message cases.

### Example 1: both lower and upper bounds supplied

Here I check whether carbon monoxide values are between 0 and 10, inclusive. This is the most standard use of the function and should create a Boolean column showing whether each observation falls in that range.

In [ ]:
obj_num_1 = SparkDataCheck.from_csv(spark, air_path)
obj_num_1.check_numeric_range("CO(GT)", lower=0, upper=10)
obj_num_1.df.select("CO(GT)", "CO(GT)_in_range").show(10)

### Example 2: lower bound only

In this example I check whether temperature is at least 0. This shows that the method works when only one side of the range is supplied.

In [ ]:
obj_num_2 = SparkDataCheck.from_csv(spark, air_path)
obj_num_2.check_numeric_range("T", lower=0)
obj_num_2.df.select("T", "T_in_range").show(10)

### Example 3: upper bound only

Here I check whether relative humidity is at most 80. Again, this uses only one side of the bound and confirms that the method still behaves correctly.

In [ ]:
obj_num_3 = SparkDataCheck.from_csv(spark, air_path)
obj_num_3.check_numeric_range("RH", upper=80)
obj_num_3.df.select("RH", "RH_in_range").show(10)

### Example 4: no bounds supplied

This case should print a message because the method needs at least one bound. The DataFrame should remain unchanged.

In [ ]:
obj_num_4 = SparkDataCheck.from_csv(spark, air_path)
obj_num_4.check_numeric_range("AH")
obj_num_4.df.show(5)

### Example 5: non-numeric column supplied

The `Date` column is not numeric, so this should trigger the message case and leave the DataFrame unchanged.

In [ ]:
obj_num_5 = SparkDataCheck.from_csv(spark, air_path)
obj_num_5.check_numeric_range("Date", lower=0, upper=100)
obj_num_5.df.show(5)

The five examples above show the full behavior of the range-check method. The successful cases add the requested Boolean flag, while the failure cases demonstrate that the method protects the object from invalid input instead of silently doing something incorrect.